# NeuroPAL whole-brain activity (Yemini et al. 2021) from dandiset 000541

Paper: Yemini, Lin, Nejatbakhsh et al. (2021), "NeuroPAL: A Multicolor Atlas for Whole-Brain Neuronal Identification in C. elegans", Cell 184, 272-288, [doi:10.1016/j.cell.2020.12.012](https://doi.org/10.1016/j.cell.2020.12.012).

Dataset: [DANDI:000541](https://dandiarchive.org/dandiset/000541/0.241009.1457), "NeuroPAL Microfluidic Chip Images and GCaMP activity", version 0.241009.1457 (doi:10.48324/dandi.000541/0.241009.1457). The paper cites no version of the dandiset (it cites its Zenodo deposit). Version 0.241009.1457 is the most recent published version; the three published versions (0.240611.1947, 0.240625.0452, 0.241009.1457) list the same 21 assets (the same asset identifiers), so none matches the claims file better, and the draft can change.

Claims file: `claims/000541/10.1016-j.cell.2020.12.012.yaml`, written from the accepted author manuscript in PubMed Central (PMC10494711) with Methods S1 and S2. Outside inputs: the paper's own deposit of the traces on Zenodo (doi:10.5281/zenodo.3906530), used for a link check and a variant, and the synapse counts of Cook et al. (2019) (doi:10.1038/s41586-019-1352-7), for the connectome claims.

The dandiset holds the 21 OH16230 head recordings of Figure 6 and nothing else: no tail recordings, no OH15500 animals, and none of the imaging behind Figures 1 to 5 and 7.

| Claim | Tested here |
|---|---|
| link-zenodo-traces | tested (link) |
| fig6-sample-sizes | tested (heads only; tails not in the dandiset) |
| fig6-all-heads-responsive | tested |
| fig6d-ase-nacl | tested |
| fig6-nacl-unknown-interneurons | tested |
| fig6c-odors-more-inhibitory | tested |
| fig6d-awc-asymmetric | blocked: the dandiset does not label AWC ON and OFF (a variant with the Zenodo labels is shown) |
| fig6-asj-ric | tested |
| fig6-interneurons-downstream | tested |
| fig6-tail-responses | blocked: no tail recordings in the dandiset |
| fig6f-pharyngeal | tested |
| fig6-stimulus-sets-distinct | tested |
| figs1h-avf-cyclic | tested |
| fig6gi-stimulus-specific-correlations | tested |
| figs2a-pca-trajectories | tested |
| link-connectome-names | tested (link) |
| fig6jk-connectome-correlation | tested (heads only) |
| fig6-connectome-variations | tested (heads only) |
| fig1, figs1, fig2, fig4, fig5 and fig7 claims (11) | blocked: the data are not in the dandiset |

In [ ]:
import os
os.environ.setdefault("OMP_NUM_THREADS", "1")
import time
import warnings
import collections
import numpy as np
import pandas as pd
from scipy import stats, signal
import matplotlib.pyplot as plt
import neuropal as npl

T_START = time.time()
warnings.filterwarnings("ignore", category=RuntimeWarning)  # empty means over animals lacking a pair
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
print("dandiset", npl.DANDISET_ID, "version", npl.DANDISET_VERSION)

In [ ]:
assets = npl.list_assets()
print(len(assets), "assets;", round(sum(a["size"] for a in assets) / 1e9, 2), "GB")
recs = npl.load_all()
recs = sorted(recs, key=lambda r: r["identifier"])
print("loaded in", round(time.time() - T_START), "s")

## Data check

What was loaded, and the checks of the description against the values.

In [ ]:
rows = []
for r in recs:
    named = [n for n in r["labels"] if n]
    allnan = int(np.sum(np.all(~np.isfinite(r["raw"]), axis=0)))
    order = sorted(r["stimuli"], key=lambda s: r["stimuli"][s][0])
    rows.append({
        "file": r["path"].split("/")[-1], "identifier": r["identifier"], "strain": r["strain"],
        "stage": r["growth_stage"], "description": r["session_description"],
        "frames": r["raw"].shape[0], "movie_frames": r["n_movie_frames"], "rate_Hz": r["rate"],
        "duration_s": r["raw"].shape[0] / r["rate"], "rois": r["raw"].shape[1], "named": len(named),
        "unnamed": r["raw"].shape[1] - len(named), "dup_names": len(named) - len(set(named)),
        "all_nan_traces": allnan, "two_series_equal": r["raw_equals_other_series"],
        "stim_order": "/".join(s[:4] for s in order),
        "stim_times": ";".join(f"{r['stimuli'][s][0]}-{r['stimuli'][s][1]}" for s in order),
        "raw_min": float(np.nanmin(r["raw"])), "raw_max": float(np.nanmax(r["raw"])),
    })
sanity = pd.DataFrame(rows)
print(sanity.drop(columns=["file"]).to_string())
print()
print("files:", len(recs), "| strains:", list(sanity.strain.unique()), "| stages:", list(sanity.stage.unique()),
      "| descriptions:", list(sanity.description.unique()))
print("trace unit:", {r["unit"] for r in recs}, "| rate:", {r["rate"] for r in recs},
      "| starting time:", {r["t0"] for r in recs})
print("grid spacing (um):", {tuple(np.round(r["grid_spacing_um"], 4)) for r in recs})
print("frames per trace: %d to %d (%.1f to %.1f s at 4 Hz)" % (sanity.frames.min(), sanity.frames.max(),
      sanity.frames.min() / 4, sanity.frames.max() / 4))
print("trace frames = movie frames - 1 in every file:", bool((sanity.frames == sanity.movie_frames - 1).all()))
print("stimulus orders:", dict(collections.Counter(sanity.stim_order)))
print("stimulus windows:", list(sanity.stim_times.unique()))
print("stimulus table description:", {r["stim_description"] for r in recs})
print("the dNMF series equals the 'raw' series (NaN equal to NaN) in", int(sanity.two_series_equal.sum()),
      "of", len(recs), "files")
print("raw trace values: %.3f to %.3f (unitless)" % (sanity.raw_min.min(), sanity.raw_max.max()))

The dandiset holds 21 files, one per animal, each described as a head of strain OH16230 at the young-adult stage. The paper's 21 tails and its OH15500 animals are not in it. Each file holds about 240 s at 4 Hz, the 4 min recording of the STAR Methods, and three 10 s stimuli starting at 60.5, 120.5 and 180.5 s, one per minute after the first minute, as the STAR Methods describe. The stimulus order takes three of the six possible orders, seven heads each, which fits the STAR Methods' "rotated periodically" better than the file's "in a random order". The table's description gives 2-butanone as "10^-4mM" and 2,3-pentanedione as "10^-4"; the paper gives both odors as 10^-4 dilutions, so the unit "mM" is likely wrong. The two trace series in each file (`SignalRawFluor` and `SignalRawFluordNMF`) are the same numbers, so only one is used.

Next: recordings deposited twice, and the names of the neurons.

In [ ]:
# Recordings deposited more than once: identical trace matrices, or any trace identical across files.
fp = {}
for r in recs:
    x = np.nan_to_num(r["raw"])
    fp[r["identifier"]] = x
same = []
ids = list(fp)
for i in range(len(ids)):
    for j in range(i + 1, len(ids)):
        a, b = fp[ids[i]], fp[ids[j]]
        n = min(len(a), len(b))
        ca = a[:n].T[np.std(a[:n], axis=0) > 0]
        cb = b[:n].T[np.std(b[:n], axis=0) > 0]
        za = (ca - ca.mean(1, keepdims=True)) / ca.std(1, keepdims=True)
        zb = (cb - cb.mean(1, keepdims=True)) / cb.std(1, keepdims=True)
        cmax = (za @ zb.T / n).max()
        if cmax > 0.999:
            same.append((ids[i], ids[j], cmax))
print("pairs of files sharing a trace correlated above 0.999:", same if same else "none")

names = sorted({n for r in recs for n in r["labels"] if n})
print("distinct neuron names over the 21 heads:", len(names))
cnt = collections.Counter(n for r in recs for n in set(r["labels"]) if n)
print("names in 3 or fewer heads:", {n: c for n, c in cnt.items() if c <= 3})
nonneuronal = [n for n in names if n in ("AMSOL", "AMSOR", "HMC")]
print("names that are not neurons (amphid sheath glia AMSO, head mesodermal cell HMC):", nonneuronal,
      {n: cnt[n] for n in nonneuronal})
print("DD6 (a motor neuron of the posterior ventral cord) named in", cnt["DD6"], "head file(s):",
      [r["identifier"] for r in recs if "DD6" in r["labels"]])

### Trace processing

The STAR Methods ("Brainwide imaging of neuronal activity traces") give the processing: remove the first 1 s and the last frame, subtract a fitted first-order exponential bleach curve, divide by the 5th percentile of the trace, and remove neurons within 5 um of each other whose traces correlate at 0.95 or more. The paper does not say which member of such a pair is removed. Its sentence ("Neurons within 5 um distance of each other, whose traces showed at least 95% correlation, were deemed too similar and removed") reads most directly as removing both, which is the main reading here, adopted as an assumption. Two more readings are run throughout: remove one of the two (the later in the file's order), and remove none. The dandiset's traces were extracted again for this analysis by the same method, so the pairs found at the 0.95 threshold need not be the paper's.

In [ ]:
data = {mode: npl.prepare(recs, mode) for mode in ("both", "one", "none")}
D = data["both"]
cls_of = D["class_of"]
for mode, d in data.items():
    kept = [set(a["traces"]) for a in d["animals"]]
    print(f"near-duplicate reading '{mode}': removed {sum(len(a['duplicates']) for a in d['animals'])} traces;"
          f" kept {sum(len(k) for k in kept)} named traces;"
          f" traces dropped as all NaN: {sum(len(a['dropped_nan']) for a in d['animals'])}")
allvals = np.concatenate([np.concatenate([t[4:] for t in a["traces"].values()]) for a in D["animals"]])
print("dF/F0 over all kept traces: %.2f to %.2f, median %.3f" % (allvals.min(), allvals.max(), np.median(allvals)))
print("classes (left and right pooled):", len(set(cls_of.values())))

A few traces have a 5th percentile near zero, which gives the extreme dF/F0 values in the range above. The tests use means over 10 s windows and keep these traces, as the paper describes no trimming.

### Clock of the stimulus table

The responses of the primary sensory neurons should line up with the stimulus table. For ASE and NaCl, and for AWC and the two odors, the response (mean during the 10 s stimulus minus the 10 s before) is computed at the table's times and with each stimulus's times moved to the other two stimulus windows of the same animal (a shift of 60 or 120 s), which keeps the clock but breaks the labels.

In [ ]:
def resp_at(a, n, s0, s1):
    dur, pre = npl.window_means(a["traces"][n], s0, s1, a["rate"])
    return dur - pre

clock = []
for cls, stims, sign in (("ASE", ["nacl"], 1), ("AWC", ["butanone", "pentanedione"], -1)):
    for s in stims:
        true, shifted = [], []
        for a in D["animals"]:
            for n in (cls + "L", cls + "R"):
                if n not in a["traces"]:
                    continue
                true.append(resp_at(a, n, *a["stimuli"][s]))
                for o in npl.STIMULI:
                    if o != s:
                        shifted.append(resp_at(a, n, *a["stimuli"][o]))
        clock.append({"neuron": cls, "stimulus": s, "true_mean": np.mean(true), "n_true": len(true),
                      "other_windows_mean": np.mean(shifted), "n_other": len(shifted),
                      "mannwhitney_p": stats.mannwhitneyu(true, shifted).pvalue})
print(pd.DataFrame(clock).round(4).to_string())

ASE responds more to NaCl at the table's NaCl time than in the other stimulus windows, and AWC falls more at the odor times than in the other windows (numbers above). This is consistent with the stimulus table sharing the clock of the traces. It does not prove it: both neurons also respond in the other windows, as every window is a stimulus of some kind (see fig6-all-heads-responsive).

## link-zenodo-traces

> The dandiset's head recordings are the 21 OH16230 heads of the paper's Zenodo deposit: files match by identifier, stimulus orders agree, and the dF/F0 computed from the dandiset's traces by the STAR Methods matches the deposit's traces.

The deposit (doi:10.5281/zenodo.3906530, `Head_Activity_OH16230.mat`) is the paper's own record of these traces; it is read here only to check the link, and for the AWC ON/OFF variant of fig6d-awc-asymmetric. Its README says traces are stored per class, left traces then right traces, each in file order; the deposit keeps one slot per file for each neuron, empty (NaN) where the file lacks it, which is checked below.

Criterion (written by the author of this attempt, as link claims are): reproduced when every dandiset file matches a deposit file by identifier, the stimulus orders agree in every file, and at least 90% of the traces present in both correlate at r of 0.9 or more; partially reproduced when files and stimuli match but fewer traces agree.

In [ ]:
Z = npl.load_zenodo_heads()
zfiles = ["2019" + str(f) for f in Z["files"]]
our_ids = [a["id"] for a in data["none"]["animals"]]
print("deposit files:", len(zfiles), "| matched to dandiset identifiers:", sum(f in our_ids for f in zfiles),
      "| same order:", zfiles == our_ids)
znames = [str(s) for s in Z["stim_names"]]
zorder = {f: [znames[k - 1] for k in np.atleast_1d(Z["stims"][i])] for i, f in enumerate(zfiles)}
ourorder = {a["id"]: sorted(a["stimuli"], key=lambda s: a["stimuli"][s][0]) for a in data["none"]["animals"]}
print("stimulus order agrees in", sum(zorder[f] == ourorder[f] for f in zfiles), "of", len(zfiles), "files")
print("deposit stimulus windows:", np.asarray(Z["stim_times"]).tolist(), "| fps:", Z["fps"])

zt = npl.zenodo_traces_by_file(Z)
NONE = {a["id"]: a for a in data["none"]["animals"]}
link_rows, count_mismatch = [], []
for cls, (trs, is_l) in zt.items():
    if np.all(np.isnan(is_l)):
        groups = [(cls, list(range(len(trs))))]
    else:
        groups = [(cls + "L", [k for k in range(len(trs)) if is_l[k] == 1]),
                  (cls + "R", [k for k in range(len(trs)) if is_l[k] == 0])]
    for name, ks in groups:
        if len(ks) != len(our_ids):
            count_mismatch.append((name, len(ks)))
            continue
        for f, k in zip(our_ids, ks):
            z = np.asarray(trs[k], float)
            in_dep = bool(np.isfinite(z).any())
            in_dan = name in NONE[f]["traces"]
            row = {"file": f, "neuron": name, "in_deposit": in_dep, "in_dandiset": in_dan, "r": np.nan,
                   "slope": np.nan, "deposit_len": len(z), "nan_head": -1}
            if in_dep and in_dan:
                ours = NONE[f]["traces"][name]
                nn = min(len(z), len(ours))
                z, ours = z[:nn], ours[:nn]
                ok = np.isfinite(z) & np.isfinite(ours)
                row.update(r=np.corrcoef(z[ok], ours[ok])[0, 1], slope=np.polyfit(ours[ok], z[ok], 1)[0],
                           nan_head=int(np.argmax(np.isfinite(z))))
            link_rows.append(row)
link = pd.DataFrame(link_rows)
deposit_names = sum(1 if np.all(np.isnan(il)) else int((il == 1).any()) + int((il == 0).any()) for _, il in zt.values())
print("deposit neuron names:", deposit_names, "| classes:", len(zt))
print("names without one slot per file in the deposit:", count_mismatch if count_mismatch else "none")
allnamed = {(a["id"], n) for a in NONE.values() for n in a["traces"]}
print("named dandiset traces:", len(allnamed), "| deposit slots with a trace:", int(link.in_deposit.sum()))
print("in both:", int((link.in_deposit & link.in_dandiset).sum()),
      "| in the deposit only:", int((link.in_deposit & ~link.in_dandiset).sum()),
      "| in the dandiset only:", int((~link.in_deposit & link.in_dandiset).sum()),
      "| dandiset traces with a name the deposit lacks:", sorted({n for f, n in allnamed} - set(link.neuron)))
only_dan = link[~link.in_deposit & link.in_dandiset]
print("dandiset-only traces by name:", only_dan.neuron.value_counts().to_dict())
link = link[link.in_deposit & link.in_dandiset].copy()
print("Pearson r between dandiset dF/F0 and deposit trace: median %.4f, 5th percentile %.4f, min %.4f"
      % (link.r.median(), link.r.quantile(0.05), link.r.min()))
print("fraction with r >= 0.9: %.3f; r >= 0.95: %.3f; r >= 0.99: %.3f"
      % ((link.r >= 0.9).mean(), (link.r >= 0.95).mean(), (link.r >= 0.99).mean()))
print("regression slope deposit on dandiset: median %.3f (IQR %.3f to %.3f)"
      % (link.slope.median(), link.slope.quantile(0.25), link.slope.quantile(0.75)))
link["len_diff"] = [row.deposit_len - len(NONE[row.file]["traces"][row.neuron]) for row in link.itertuples()]
print("deposit trace length minus dandiset frames (after dropping the last frame):",
      link.groupby("file").len_diff.first().value_counts().to_dict(),
    "| leading NaN frames in the deposit:", sorted(link.nan_head.unique()))
print(link.groupby("file").r.agg(["count", "median", "min"]).round(4).to_string())

**Result: partially reproduced.** All 21 dandiset files match a deposit file by identifier, in the same order, and the stimulus order agrees in all 21. The deposit has one slot per file for each of its 189 neuron names, so traces pair by file. Of the 3,663 named dandiset traces, 3,550 are also in the deposit. 101 are in the dandiset only (11 of them AVL, which the deposit leaves empty in every head), and 4 names (AMSOL, AMSOR, HMC, DD6) are not in the deposit at all. dF/F0 computed from the dandiset by the STAR Methods correlates with the deposit's traces at a median r of 0.972, with a median slope of 0.985. 81.7% of traces reach r of 0.9, below the 90% the criterion asks for. The deposit's traces have their first 4 frames (1 s) empty and are one frame shorter than the dandiset's, or one frame longer, in 10 of the 21 files.

## fig6-sample-sizes

> Whole-brain activity was recorded in 21 heads (189 identified neurons) and, separately, 21 tails (42 neurons), with a median of 18 animals per neuron type.

Readings from the criterion: neurons as (a) distinct names identified in at least one head, (b) the same after the STAR Methods removals (near-duplicate traces, N of 3 or fewer); neuron type as (a) a name, (b) a class. The removals are run under each near-duplicate reading. The dandiset holds no tails, so the tail counts are 0 here.

In [ ]:
ss = []
for mode in ("both", "one", "none"):
    d = data[mode]
    kept = [set(a["traces"]) for a in d["animals"]]
    by_name = collections.Counter(n for k in kept for n in k)
    by_cls = collections.Counter()
    for k in kept:
        by_cls.update({cls_of[n] for n in k})
    named_any = {n for r in recs for n in r["labels"] if n}
    ss.append({"duplicates": mode, "heads": len(d["animals"]),
               "names_any_head": len(named_any),
               "names_any_head_neurons_only": len(named_any - {"AMSOL", "AMSOR", "HMC"}),
               "names_after_removals": sum(v > npl.MIN_N for v in by_name.values()),
               "median_animals_per_name": float(np.median(list(by_name.values()))),
               "median_animals_per_name_N>3": float(np.median([v for v in by_name.values() if v > npl.MIN_N])),
               "median_animals_per_class": float(np.median(list(by_cls.values()))),
               "median_animals_per_class_N>3": float(np.median([v for v in by_cls.values() if v > npl.MIN_N]))})
ss = pd.DataFrame(ss)
print(ss.to_string())
print("tail recordings in the dandiset:", sum("tail" in r["session_description"].lower() for r in recs))
print("expected: heads 21 (tol 0), head neurons 189 (170.1 to 207.9), tails 21, tail neurons 42, median 18 (16.2 to 19.8)")

**Result: partially reproduced.** The head counts match: 21 heads (expected 21); 193 neuron names in at least one head, 190 of them neurons, and 184 after the removals, all within 170.1 to 207.9. The median number of animals per name is 17 under the main reading (both members of a near-duplicate pair removed), 18 when one member is removed, and 21 with none removed. Per class it is 19, 20 and 21. The main reading falls within 16.2 to 19.8 for both names and classes. The dandiset holds no tail recordings, so the tail counts (21 tails, 42 neurons) cannot be matched.

## fig6-all-heads-responsive

> 21 head recordings showed the published responses of the primary sensory neurons, ASE excitation to NaCl and AWC inhibition to both odors, in the left or the right neuron, and were kept for analysis.

Readings: (a) the mean dF/F0 during the 10 s pulse exceeds (ASE) or falls below (AWC) the mean of the 10 s before; (b) the difference exceeds two standard deviations of the 10 s before. Near-duplicate removal is not applied here (reading 'none'), as the paper's review of the heads precedes its statistics; the main reading is shown as a variant.

In [ ]:
def head_checks(d, k_sd):
    out = []
    for a in d["animals"]:
        row = {"animal": a["id"]}
        for cls, stim, sign in (("ASE", "nacl", 1), ("AWC", "butanone", -1), ("AWC", "pentanedione", -1)):
            ok = False
            for n in (cls + "L", cls + "R"):
                if n not in a["traces"]:
                    continue
                tr = a["traces"][n]
                s0, s1 = a["stimuli"][stim]
                t = np.arange(len(tr)) / a["rate"]
                pre = tr[(t >= s0 - 10) & (t < s0)]
                diff = tr[(t >= s0) & (t < s1)].mean() - pre.mean()
                ok |= sign * diff > k_sd * pre.std()
            row[f"{cls}_{stim}"] = ok
        row["all_three"] = all(v for k, v in row.items() if k != "animal")
        out.append(row)
    return pd.DataFrame(out)

for label, mode, k in (("(a) none-removed", "none", 0), ("(b) none-removed", "none", 2),
                       ("(a) both-removed", "both", 0), ("(b) both-removed", "both", 2)):
    hc = head_checks(data[mode], k)
    print(label, "heads passing all three:", int(hc.all_three.sum()), "of", len(hc),
          "| per stimulus:", hc.drop(columns=["animal", "all_three"]).sum().to_dict())
hc_a = head_checks(data["none"], 0)
print(hc_a.to_string())

**Result: not reproduced.** Under reading (a), the more lenient one, 19 of the 21 heads pass all three checks when no neuron is removed: ASE to NaCl in 21, AWC to 2-butanone in 20, AWC to 2,3-pentanedione in 19. Under reading (b) 16 pass. With near-duplicates removed, 18 pass under (a) and 15 under (b). Head 20191030_07 fails on 2,3-pentanedione and head 20191104_10 on both odors. The dandiset holds exactly the 21 heads the paper kept, so the paper's review, which it does not describe, passed two heads that the claims file's rules do not.

### The response tests used by the claims below

Per the STAR Methods: for each class (left and right pooled), each stimulus, a two-tailed paired t-test of the mean dF/F0 during the 10 s stimulus against the identical period immediately before; premotor interneurons and ventral-cord motor neurons excluded; classes in 3 or fewer animals not evaluated. q values by Storey (2002). The FDR family has two readings: (a) all class and stimulus tests of the heads together, (b) each stimulus apart; (c) uncorrected p is the third reading where the claims file lists it. The table is built for each near-duplicate reading.

In [ ]:
def test_table(d):
    rows = []
    for s in npl.STIMULI:
        for c, v in npl.class_tests(npl.responses(d, s)).items():
            rows.append({"stimulus": s, "cls": c, **v})
    t = pd.DataFrame(rows)
    t["q_all"] = npl.storey_q(t.p.values)
    t["q_stim"] = np.nan
    for s in npl.STIMULI:
        m = t.stimulus == s
        t.loc[m, "q_stim"] = npl.storey_q(t.loc[m, "p"].values)
    t["q_bh_all"] = npl.bh_q(t.p.values)
    return t

TT = {mode: test_table(data[mode]) for mode in ("both", "one", "none")}
T = TT["both"]
print("tests per reading:", {m: len(t) for m, t in TT.items()}, "| classes tested:", T.cls.nunique())
print("Storey pi0, all tests:", round(npl.storey_pi0(T.p.values), 3),
      "| per stimulus:", {s: round(npl.storey_pi0(T[T.stimulus == s].p.values), 3) for s in npl.STIMULI})
READINGS = {"q_all": "(a) q over all tests", "q_stim": "(b) q per stimulus", "p": "(c) uncorrected p"}

def sig(t, cls, stim, key):
    r = t[(t.cls == cls) & (t.stimulus == stim)]
    if r.empty:
        return None
    return bool(r[key].iloc[0] <= 0.05), float(r.mean_diff.iloc[0])

def show(t, classes):
    sub = t[t.cls.isin(classes)].set_index(["cls", "stimulus"])[["n_traces", "n_animals", "mean_diff", "t", "p", "q_all", "q_stim"]]
    print(sub.round(5).to_string())

## fig6d-ase-nacl

> NaCl evokes left/right asymmetric responses in the two ASE neurons, and ASER shows an increase in calcium upon NaCl presentation.

ASER alone, paired t-test over heads, with its p added to the FDR family (the family's tests pool left and right; adding the one-sided test is a choice made here). Asymmetry: two-tailed unpaired t-test of ASEL against ASER, on (a) the response (during minus before) or (b) the mean during the pulse.

In [ ]:
def side_values(d, name, stim):
    out = []
    for a in d["animals"]:
        if name in a["traces"]:
            dur, pre = npl.window_means(a["traces"][name], *a["stimuli"][stim], a["rate"])
            out.append((a["id"], dur, pre))
    return out

for mode in ("both", "one", "none"):
    d, t = data[mode], TT[mode]
    L = side_values(d, "ASEL", "nacl")
    R = side_values(d, "ASER", "nacl")
    rd = np.array([x[1] for x in R]); rb = np.array([x[2] for x in R])
    pr = stats.ttest_rel(rd, rb).pvalue
    q_all = npl.storey_q(np.r_[t.p.values, pr])[-1]
    q_st = npl.storey_q(np.r_[t[t.stimulus == "nacl"].p.values, pr])[-1]
    ld = np.array([x[1] for x in L]); lb = np.array([x[2] for x in L])
    asym_resp = stats.ttest_ind(ld - lb, rd - rb)
    asym_mean = stats.ttest_ind(ld, rd)
    print(f"[{mode}] ASER NaCl: n={len(R)}, mean during-before={np.mean(rd - rb):+.3f}, "
          f"rising in {np.sum(rd > rb)} of {len(R)}, p={pr:.3g}, q(a)={q_all:.3g}, q(b)={q_st:.3g}")
    print(f"[{mode}] ASEL NaCl: n={len(L)}, mean during-before={np.mean(ld - lb):+.3f}, rising in {np.sum(ld > lb)} of {len(L)}")
    print(f"[{mode}] asymmetry (a) responses: t={asym_resp.statistic:.3f}, p={asym_resp.pvalue:.3g};"
          f" (b) means during: t={asym_mean.statistic:.3f}, p={asym_mean.pvalue:.3g}")

**Result: reproduced.** Under the main reading, ASER rises during NaCl (mean during minus before +0.502, rising in 10 of 15 heads, p = 0.020), with q = 0.026 when the FDR family is the NaCl tests (reading b) and q = 0.0505 over all tests (reading a). With one or no member of near-duplicate pairs removed, ASER keeps all 21 heads (p = 0.0055, q = 0.018 and 0.013 under reading a). ASEL and ASER differ: t = 4.53, p = 8.8e-05 on the responses, p = 3.6e-05 on the means during the pulse. ASEL responds more strongly than ASER (+1.82, rising in 17 of 17).

## fig6-nacl-unknown-interneurons

> NaCl evokes significant responses in six interneurons with no previously known function, AIN, AVF, AVH, AVJ, I2 and MI.

In [ ]:
SIX = ["AIN", "AVF", "AVH", "AVJ", "I2", "MI"]
show(T[T.stimulus == "nacl"], SIX)
for mode, t in TT.items():
    for key, lab in READINGS.items():
        res = {c: sig(t, c, "nacl", key) for c in SIX}
        print(f"[{mode}] {lab}: significant {sum(v[0] for v in res.values())} of 6:",
              [c for c, v in res.items() if v[0]])

**Result: partially reproduced.** Four of the six classes respond significantly to NaCl under every FDR reading and every near-duplicate reading: AIN (q 0.0033 over all tests), AVH (0.0053), I2 (0.0034) and MI (0.020). AVJ rises without significance (p = 0.094) and AVF does not respond (mean -0.08, p = 0.70).

## fig6c-odors-more-inhibitory

> Both odors evoke responses in many sensory and interneurons throughout the brain, with more inhibitory (decreasing calcium) activity than NaCl.

Readings: (a) the number of head classes with a significant decrease (q of 0.05 or less, under each FDR reading); (b) the fraction of head classes whose mean response is negative.

In [ ]:
for mode, t in TT.items():
    rows = []
    for s in npl.STIMULI:
        ts = t[t.stimulus == s]
        row = {"stimulus": s, "classes": len(ts), "frac_negative_mean": round((ts.mean_diff < 0).mean(), 3)}
        for key in READINGS:
            row[f"down_{key}"] = int(((ts[key] <= 0.05) & (ts.mean_diff < 0)).sum())
            row[f"up_{key}"] = int(((ts[key] <= 0.05) & (ts.mean_diff > 0)).sum())
        rows.append(row)
    print(f"[{mode}]")
    print(pd.DataFrame(rows).to_string(index=False))

**Result: reproduced.** Under reading (a), with q over all tests, NaCl gives significant decreases in 3 of 95 head classes, 2-butanone in 5 and 2,3-pentanedione in 7. Under reading (b), 25.3% of classes have a negative mean response to NaCl, 27.4% to 2-butanone and 44.2% to 2,3-pentanedione. Both odors exceed NaCl under both readings and every near-duplicate reading. The odors evoke significant responses in 20 classes each, against 21 for NaCl.

## fig6d-awc-asymmetric (blocked)

> 2-butanone and 2,3-pentanedione evoke stochastically asymmetric responses in the AWC ON and AWC OFF neurons, AWC OFF being the one srsx-3 colors in NeuroPAL.

The criterion requires the dataset to label AWC ON and AWC OFF, or AWCL and AWCR with the ON/OFF side. The files name only AWCL and AWCR, so the claim is blocked. The paper's Zenodo deposit gives the side (`is_AWCL_OFF`); as a variant only, the test of the criterion is run with that label on the dandiset's traces.

In [ ]:
awc_terms = [k for r in recs[:1] for k in ("AWCON", "AWCOFF", "AWC_ON", "AWC_OFF") if k in r["labels"]]
print("AWC names in the files:", sorted({n for r in recs for n in r["labels"] if n.startswith("AWC")}),
      "| ON/OFF names:", awc_terms)
is_off = dict(zip(zfiles, np.asarray(Z["is_AWCL_OFF"], float)))
print("deposit: AWCL is OFF in", int(np.nansum(list(is_off.values()))), "of", len(is_off), "heads")
for mode in ("both", "one", "none"):
    d = data[mode]
    for s in ("butanone", "pentanedione"):
        on, off = {"resp": [], "mean": []}, {"resp": [], "mean": []}
        for a in d["animals"]:
            for side in "LR":
                n = "AWC" + side
                if n not in a["traces"]:
                    continue
                dur, pre = npl.window_means(a["traces"][n], *a["stimuli"][s], a["rate"])
                tgt = off if (is_off[a["id"]] == 1) == (side == "L") else on
                tgt["resp"].append(dur - pre)
                tgt["mean"].append(dur)
        for meas in ("resp", "mean"):
            r = stats.ttest_ind(on[meas], off[meas])
            print(f"[{mode}] variant, {s}, {meas}: ON {np.mean(on[meas]):+.3f} (n={len(on[meas])}) vs OFF "
                  f"{np.mean(off[meas]):+.3f} (n={len(off[meas])}), t={r.statistic:.2f}, p={r.pvalue:.3g}")

**Result: blocked (metadata insufficient).** The files name only AWCL and AWCR and do not say which is AWC ON or OFF. With the deposit's `is_AWCL_OFF` as a variant, AWC ON and OFF responses do not differ significantly: 2-butanone p = 0.56 and 2,3-pentanedione p = 0.087 on the responses (0.51 and 0.096 on the means).

## fig6-asj-ric

> The ASJ sensory neurons are excited by NaCl but inhibited by both odors, whereas the RIC interneurons are excited only by 2,3-pentanedione.

Six parts: ASJ excited by NaCl, inhibited by 2-butanone, inhibited by 2,3-pentanedione (each significant); RIC excited by 2,3-pentanedione (significant), not significantly excited by NaCl, not by 2-butanone.

In [ ]:
show(T, ["ASJ", "RIC"])
for mode, t in TT.items():
    for key, lab in READINGS.items():
        def part(cls, stim, want):
            s, m = sig(t, cls, stim, key)
            if want == "up":
                return s and m > 0
            if want == "down":
                return s and m < 0
            return not (s and m > 0)
        parts = {"ASJ+NaCl": part("ASJ", "nacl", "up"), "ASJ-but": part("ASJ", "butanone", "down"),
                 "ASJ-pent": part("ASJ", "pentanedione", "down"), "RIC+pent": part("RIC", "pentanedione", "up"),
                 "RIC not+NaCl": part("RIC", "nacl", "not_up"), "RIC not+but": part("RIC", "butanone", "not_up")}
        print(f"[{mode}] {lab}: {sum(parts.values())} of 6 hold:", parts)

**Result: partially reproduced, resting on the near-duplicate reading.** Under the main reading 4 of the 6 parts hold with q (both FDR readings) and 5 with uncorrected p. ASJ is excited by NaCl and inhibited by 2-butanone, but not significantly inhibited by 2,3-pentanedione. RIC is not excited by NaCl or 2-butanone, and its excitation by 2,3-pentanedione (p = 0.027) misses q of 0.05 (0.054 and 0.057). With one or no member of near-duplicate pairs removed, ASJ loses its NaCl response (p = 0.83) and 3 of 6 parts hold, which would make the claim not reproduced.

## fig6-interneurons-downstream

> Downstream of the sensory neurons many interneurons respond significantly, many with no previously known function, for example AIN, AVH, RIF, RIG and RIR.

A class counts when it responds significantly to at least one of the three stimuli.

In [ ]:
FIVE = ["AIN", "AVH", "RIF", "RIG", "RIR"]
show(T, FIVE)
for mode, t in TT.items():
    for key, lab in READINGS.items():
        res = {c: [s[:4] for s in npl.STIMULI if sig(t, c, s, key) and sig(t, c, s, key)[0]] for c in FIVE}
        print(f"[{mode}] {lab}: {sum(bool(v) for v in res.values())} of 5 respond:", res)

**Result: reproduced.** All five classes respond significantly to at least one stimulus under every FDR reading and every near-duplicate reading: AIN to NaCl, AVH to NaCl and 2,3-pentanedione, RIF and RIG to 2,3-pentanedione (both decreasing), and RIR to 2-butanone.

## fig6-tail-responses (blocked)

> In the tail, PVQ is significantly excited by 2,3-pentanedione, and a mix of sensory and interneurons show significant post-stimulus responses to all three stimuli, including LUA, PVN, PVQ, PVR and PVW.

The dandiset holds no tail recordings (data check above: every file is a head), so the claim cannot be tested on it. The tails are in the paper's Zenodo deposit (`Tail_Activity_OH16230.mat`).

In [ ]:
tail_names = [n for r in recs for n in r["labels"] if n[:3] in ("LUA", "PVN", "PVQ", "PVR", "PVW")]
print("tail neurons named in the dandiset:", tail_names if tail_names else "none",
      "| session descriptions:", sorted({r["session_description"] for r in recs}))

## fig6f-pharyngeal

> Both salt and odors elicit responses across the pharyngeal nervous system.

Pharyngeal classes from the criterion: I1 to I6, M1 to M5, MC, MI and NSM. Reproduced when at least two respond significantly to NaCl and at least two to one of the odors.

In [ ]:
PHAR = ["I1", "I2", "I3", "I4", "I5", "I6", "M1", "M2", "M3", "M4", "M5", "MC", "MI", "NSM"]
print("pharyngeal classes tested:", sorted(set(T.cls) & set(PHAR)), "| not tested:", sorted(set(PHAR) - set(T.cls)))
show(T, PHAR)
for mode, t in TT.items():
    for key, lab in READINGS.items():
        res = {s: [c for c in PHAR if sig(t, c, s, key) and sig(t, c, s, key)[0]] for s in npl.STIMULI}
        print(f"[{mode}] {lab}:", {s: (len(v), v) for s, v in res.items()})

**Result: reproduced.** All 14 pharyngeal classes are tested. Under the main reading with q over all tests, 3 respond significantly to NaCl (I2, I3, MI), 4 to 2-butanone (I2, I3, MC, NSM) and 2 to 2,3-pentanedione (I2, MC). With q per stimulus NaCl reaches 5 classes. Every reading gives at least two classes for NaCl and for an odor.

## fig6-stimulus-sets-distinct

> The sets of neurons activated by the three stimuli are partly overlapping but distinct for each stimulus.

Head classes significant for each stimulus; reproduced when each pair of stimuli shares at least one class and no two stimuli have the same set.

In [ ]:
for mode, t in TT.items():
    for key, lab in READINGS.items():
        sets = {s: set(t[(t.stimulus == s) & (t[key] <= 0.05)].cls) for s in npl.STIMULI}
        pairs = {f"{a[:4]}&{b[:4]}": (len(sets[a] & sets[b]), sets[a] == sets[b])
                 for i, a in enumerate(npl.STIMULI) for b in npl.STIMULI[i + 1:]}
        print(f"[{mode}] {lab}: sizes", {s[:4]: len(v) for s, v in sets.items()},
              "| shared, identical:", pairs, "| in all three:", len(set.intersection(*sets.values())))

**Result: reproduced.** Under the main reading with q over all tests, 21 classes respond to NaCl, 20 to 2-butanone and 20 to 2,3-pentanedione. The pairs share 10, 9 and 12 classes, no two sets are the same, and 7 classes respond to all three. Every reading gives the same pattern.

## figs1h-avf-cyclic

> In many animals the AVF interneurons show robust cyclical activity of about 0.3 Hz for the whole 4 min recording, whatever the stimulus, similar to the crawling frequency of freely moving worms of this strain.

Per the Figure S1H legend: traces median filtered at 0.5 s (2 frames at 4 Hz, MATLAB `medfilt1` convention), frequency as the number of peaks per recording. Peaks: local maxima with prominence of at least (a) 1 or (b) 0.5 standard deviations of the filtered trace. Animals: (a) every head with AVF identified; (b) those whose AVF trace has its largest spectral peak (Welch, 256-frame segments) between 0.1 and 1 Hz. The crawling frequency is not in the dandiset.

In [ ]:
def avf_rows(d):
    rows = []
    for a in d["animals"]:
        for n in ("AVFL", "AVFR"):
            if n not in a["traces"]:
                continue
            x = a["traces"][n][4:]
            y = npl.medfilt_matlab(x, int(round(0.5 * a["rate"])))
            dur = len(y) / a["rate"]
            f, pxx = signal.welch(y - y.mean(), fs=a["rate"], nperseg=256)
            fpk = f[1:][np.argmax(pxx[1:])]
            row = {"animal": a["id"], "neuron": n, "duration_s": dur, "spectral_peak_Hz": fpk,
                   "spectral_ok": 0.1 <= fpk <= 1.0}
            for k in (1.0, 0.5):
                pk, _ = signal.find_peaks(y, prominence=k * y.std())
                row[f"freq_{k}sd"] = len(pk) / dur
            rows.append(row)
    return pd.DataFrame(rows)

for mode in ("both", "one", "none"):
    av = avf_rows(data[mode])
    print(f"[{mode}] AVF traces: {len(av)} from {av.animal.nunique()} heads; spectral peak in 0.1-1 Hz: {int(av.spectral_ok.sum())}")
    for k in ("1.0", "0.5"):
        col = f"freq_{k}sd"
        print(f"[{mode}] peaks at {k} SD: median over all {av[col].median():.3f} Hz "
              f"(range {av[col].min():.3f} to {av[col].max():.3f});"
              f" over spectral-peak traces {av[av.spectral_ok][col].median():.3f} Hz (n={int(av.spectral_ok.sum())})")
    print(f"[{mode}] median spectral peak {av.spectral_peak_Hz.median():.3f} Hz")
av = avf_rows(data["both"])
print(av.round(3).to_string())
fig, axs = plt.subplots(5, 1, figsize=(9, 8), sharex=True)
for ax, a in zip(axs, [a for a in D["animals"] if "AVFL" in a["traces"]][:5]):
    for n, col in (("AVFL", "tab:blue"), ("AVFR", "tab:red")):
        if n in a["traces"]:
            x = a["traces"][n][4:]
            ax.plot(np.arange(len(x)) / 4 + 1, npl.medfilt_matlab(x, 2), color=col, lw=0.7)
    ax.set_ylabel(a["id"][-7:], fontsize=7)
axs[-1].set_xlabel("time (s)")
plt.tight_layout()
plt.show()

**Result: not reproduced.** Under the main reading, 29 AVF traces from 18 heads give a median of 0.121 Hz with peaks at 1 SD of prominence (0.046 to 0.268 Hz) and 0.192 Hz at 0.5 SD (0.104 to 0.301 Hz). Neither is within 0.27 to 0.33 Hz. Only 3 traces have their largest spectral peak between 0.1 and 1 Hz (median 0.17 and 0.18 Hz); the rest peak at 0.016 Hz, the lowest frequency resolved. Every near-duplicate reading gives the same.

## fig6gi-stimulus-specific-correlations

> Pairwise correlations between neurons are similar across individuals for a stimulus, but each stimulus generates its own brainwide correlation pattern, even the two attractive odors.

Per the Figure 6 legend, Pearson correlations between head neurons in each animal over the 30 s following stimulus onset. Readings: (a) for each pair of animals, the correlation between their correlation matrices over shared neuron pairs, same stimulus against different stimuli, one-sided Wilcoxon signed-rank test over animal pairs; (b) 100 random splits of the animals into halves, comparing the correlation of the halves' mean matrices for the same stimulus with that for different stimuli.

In [ ]:
def corr_mats(d, window=30.0):
    allnames = sorted({n for a in d["animals"] for n in a["traces"]})
    ix = {n: i for i, n in enumerate(allnames)}
    M = {}
    for a in d["animals"]:
        names = sorted(a["traces"])
        X = np.array([a["traces"][n] for n in names]).T
        t = np.arange(len(X)) / a["rate"]
        for s in npl.STIMULI:
            s0 = a["stimuli"][s][0]
            C = np.corrcoef(X[(t >= s0) & (t < s0 + window)].T)
            full = np.full((len(allnames), len(allnames)), np.nan)
            ii = [ix[n] for n in names]
            full[np.ix_(ii, ii)] = C
            M[(a["id"], s)] = full
    return M, allnames

def mat_sim(A, B):
    iu = np.triu_indices_from(A, 1)
    a, b = A[iu], B[iu]
    ok = np.isfinite(a) & np.isfinite(b)
    return np.corrcoef(a[ok], b[ok])[0, 1]

SPAIRS = [("nacl", "butanone"), ("nacl", "pentanedione"), ("butanone", "pentanedione")]
for mode in ("both", "one", "none"):
    M, allnames = corr_mats(data[mode])
    ids = [a["id"] for a in data[mode]["animals"]]
    sims = {}
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            for x in npl.STIMULI:
                for y in npl.STIMULI:
                    sims[(i, j, x, y)] = mat_sim(M[(ids[i], x)], M[(ids[j], y)])
    for x, y in SPAIRS:
        same = np.array([(sims[(i, j, x, x)] + sims[(i, j, y, y)]) / 2 for i in range(len(ids)) for j in range(i + 1, len(ids))])
        diff = np.array([(sims[(i, j, x, y)] + sims[(i, j, y, x)]) / 2 for i in range(len(ids)) for j in range(i + 1, len(ids))])
        w = stats.wilcoxon(same, diff, alternative="greater")
        print(f"[{mode}] (a) {x[:4]}/{y[:4]}: same {np.mean(same):.4f}, different {np.mean(diff):.4f}, "
              f"same > different in {np.sum(same > diff)} of {len(same)} animal pairs, Wilcoxon p={w.pvalue:.3g}")
    print(f"[{mode}] (a) mean similarity between animals, same stimulus:",
          {x[:4]: round(np.mean([sims[(i, j, x, x)] for i in range(len(ids)) for j in range(i + 1, len(ids))]), 4) for x in npl.STIMULI})
    # (b) random halves
    rng = np.random.default_rng(611)
    wins = collections.Counter()
    gaps = collections.defaultdict(list)
    for rep in range(100):
        perm = rng.permutation(len(ids))
        h1, h2 = perm[: len(ids) // 2], perm[len(ids) // 2:]
        with np.errstate(all="ignore"):
            mean1 = {s: np.nanmean([M[(ids[k], s)] for k in h1], axis=0) for s in npl.STIMULI}
            mean2 = {s: np.nanmean([M[(ids[k], s)] for k in h2], axis=0) for s in npl.STIMULI}
        for x, y in SPAIRS:
            same = (mat_sim(mean1[x], mean2[x]) + mat_sim(mean1[y], mean2[y])) / 2
            diff = (mat_sim(mean1[x], mean2[y]) + mat_sim(mean1[y], mean2[x])) / 2
            wins[(x, y)] += same > diff
            gaps[(x, y)].append(same - diff)
    for x, y in SPAIRS:
        print(f"[{mode}] (b) {x[:4]}/{y[:4]}: same > different in {wins[(x, y)]} of 100 splits, "
              f"mean gap {np.mean(gaps[(x, y)]):.4f} (range {np.min(gaps[(x, y)]):.4f} to {np.max(gaps[(x, y)]):.4f})")

The same comparison with a window without stimulus (30 s starting 30 s after each stimulus onset, ending before the next stimulus), on the main reading, is a null: if same-window similarity is higher there too, part of the effect comes from the time in the recording or the order of the stimuli, not the stimulus itself.

In [ ]:
def shifted_mats(d, offset=30.0, window=30.0):
    allnames = sorted({n for a in d["animals"] for n in a["traces"]})
    ix = {n: i for i, n in enumerate(allnames)}
    M = {}
    for a in d["animals"]:
        names = sorted(a["traces"])
        X = np.array([a["traces"][n] for n in names]).T
        t = np.arange(len(X)) / a["rate"]
        for s in npl.STIMULI:
            s0 = a["stimuli"][s][0] + offset
            C = np.corrcoef(X[(t >= s0) & (t < s0 + window)].T)
            full = np.full((len(allnames), len(allnames)), np.nan)
            ii = [ix[n] for n in names]
            full[np.ix_(ii, ii)] = C
            M[(a["id"], s)] = full
    return M

Mn = shifted_mats(D)
ids = [a["id"] for a in D["animals"]]
for x, y in SPAIRS:
    same, diff = [], []
    for i in range(len(ids)):
        for j in range(i + 1, len(ids)):
            same.append((mat_sim(Mn[(ids[i], x)], Mn[(ids[j], x)]) + mat_sim(Mn[(ids[i], y)], Mn[(ids[j], y)])) / 2)
            diff.append((mat_sim(Mn[(ids[i], x)], Mn[(ids[j], y)]) + mat_sim(Mn[(ids[i], y)], Mn[(ids[j], x)])) / 2)
    same, diff = np.array(same), np.array(diff)
    print(f"[both] null window {x[:4]}/{y[:4]}: same {same.mean():.4f}, different {diff.mean():.4f}, "
          f"Wilcoxon p={stats.wilcoxon(same, diff, alternative='greater').pvalue:.3g}")
rng = np.random.default_rng(612)
wins = collections.Counter()
for rep in range(100):
    perm = rng.permutation(len(ids))
    h1, h2 = perm[: len(ids) // 2], perm[len(ids) // 2:]
    with np.errstate(all="ignore"):
        m1 = {s: np.nanmean([Mn[(ids[k], s)] for k in h1], axis=0) for s in npl.STIMULI}
        m2 = {s: np.nanmean([Mn[(ids[k], s)] for k in h2], axis=0) for s in npl.STIMULI}
    for x, y in SPAIRS:
        wins[(x, y)] += (mat_sim(m1[x], m2[x]) + mat_sim(m1[y], m2[y])) / 2 > (mat_sim(m1[x], m2[y]) + mat_sim(m1[y], m2[x])) / 2
print("[both] null window (b): same > different in", {f"{x[:4]}/{y[:4]}": wins[(x, y)] for x, y in SPAIRS}, "of 100 splits")

**Result: reproduced, resting on the near-duplicate reading; the null window shows the same pattern.** Under the main reading and reading (b), same-stimulus similarity exceeds different-stimulus similarity in 98 of 100 splits for each of the three stimulus pairs. Under reading (a) it does so for NaCl against 2-butanone (p = 4.4e-06) and NaCl against 2,3-pentanedione (p = 4.0e-05), but not between the two odors (p = 0.34). With one member removed, reading (b) gives 91, 97 and 94 splits; with none removed, 56, 82 and 71. Those readings support only the partial outcome, through reading (a). In the null window, 30 s after each onset with no stimulus, reading (b) also gives 100 of 100 splits for NaCl against 2,3-pentanedione and for the two odors. The test therefore does not separate stimulus-specific correlations from differences between the three periods of the recording. The similarities are small: the mean correlation between two animals' matrices for the same stimulus is 0.041 to 0.052.

## figs2a-pca-trajectories

> The average trajectories of whole-brain activity in the space of its first three principal components, over the 10 s stimulus and 20 s after, show similarities but remain distinguishable between the three stimuli.

Classes present in all 21 heads (class-averaged dF/F0 per head). PCA (a) on the whole recordings of all heads, (b) on the stimulus windows only. Trajectories are the 30 s from stimulus onset projected on the first three components, averaged over the 21 heads. Two trajectories are distinguishable when their mean distance exceeds twice the mean standard error (the norm of the per-component SEM, averaged over time and the two stimuli).

In [ ]:
def class_traces(d):
    out = {}
    for a in d["animals"]:
        g = collections.defaultdict(list)
        for n, tr in a["traces"].items():
            g[cls_of[n]].append(tr)
        out[a["id"]] = {c: np.mean(v, axis=0) for c, v in g.items()}
    return out

for mode, zscore in (("both", False), ("one", False), ("none", False), ("both", True)):
    CT = class_traces(data[mode])
    if zscore:
        CT = {k: {c: (v - np.nanmean(v)) / np.nanstd(v) for c, v in d_.items()} for k, d_ in CT.items()}
    common = sorted(set.intersection(*[set(v) for v in CT.values()]))
    nfr = min(len(next(iter(v.values()))) for v in CT.values())
    A = {k: np.array([v[c][:nfr] for c in common]).T for k, v in CT.items()}
    stim_win = {}
    for a in data[mode]["animals"]:
        t = np.arange(nfr) / a["rate"]
        for s in npl.STIMULI:
            s0 = a["stimuli"][s][0]
            stim_win[(a["id"], s)] = A[a["id"]][(t >= s0) & (t < s0 + 30)][:120]
    for reading in ("a", "b"):
        if reading == "a":
            X = np.vstack([A[k][4:] for k in A])
            X = X[np.all(np.isfinite(X), axis=1)]
        else:
            X = np.vstack(list(stim_win.values()))
            X = X[np.all(np.isfinite(X), axis=1)]
        mu = X.mean(0)
        U, S, Vt = np.linalg.svd(X - mu, full_matrices=False)
        ev = S ** 2 / np.sum(S ** 2)
        P = {k: (w - mu) @ Vt[:3].T for k, w in stim_win.items()}
        mean = {s: np.mean([P[(a, s)] for a in A], axis=0) for s in npl.STIMULI}
        sem = {s: np.linalg.norm(stats.sem([P[(a, s)] for a in A], axis=0), axis=1) for s in npl.STIMULI}
        res = []
        for x, y in SPAIRS:
            dist = np.linalg.norm(mean[x] - mean[y], axis=1).mean()
            thr = 2 * np.mean([sem[x].mean(), sem[y].mean()])
            res.append(f"{x[:4]}/{y[:4]} dist {dist:.3f} vs 2xSEM {thr:.3f} -> {'distinguishable' if dist > thr else 'not'}")
        print(f"[{mode}{', z-scored variant' if zscore else ''}] ({reading}) {len(common)} classes, variance explained by PC1-3: {np.round(ev[:3], 3)}; " + "; ".join(res))

**Result: not reproduced.** Under the main reading, 28 classes are present in all 21 heads. Reading (a) separates none of the three pairs by the criterion's rule. Reading (b) separates NaCl from 2,3-pentanedione only (distance 0.859 against 0.815). With one or no member removed (48 and 81 classes), each reading separates at most one pair, and the z-scored variant also one. No reading separates all three.

## link-connectome-names

> The dandiset's neuron names can be matched to the cell names of the Cook et al. (2019) hermaphrodite connectome.

Names are matched as written, after removing the zero padding of motor neuron numbers in Cook et al. (VA01 is VA1). Reproduced when every name that denotes a neuron matches.

In [ ]:
chem, gap = npl.load_cook()
print("Cook et al. synapse-count matrices: chemical", chem.shape, "total", chem.values.sum(), "| gap junction", gap.shape,
      "total", gap.values.sum(), "| gap matrix symmetric:", bool(np.allclose(gap.values, gap.values.T)))
dnames = sorted({n for r in recs for n in r["labels"] if n})
unmatched = [n for n in dnames if n not in chem.index or n not in gap.index]
print("dandiset names:", len(dnames), "| matched:", len(dnames) - len(unmatched), "| unmatched:", unmatched)

**Result: reproduced.** 190 of the 193 dandiset names match a cell of the Cook et al. synapse-count matrices. The three that do not, AMSOL, AMSOR and HMC, are not neurons.

## fig6jk-connectome-correlation

> Functional correlations correspond little to the anatomical connectome; the R2 of the Pearson correlation between absolute functional correlations and synapse counts is 2.8% (head) and 1.7% (tail) for electrical synapses, and 0.5% (head) and 1.5% (tail) for chemical synapses.

Head only; the dandiset has no tails. Readings: functional matrix (a) per-animal correlation over the whole recording, averaged over animals, (b) per-animal pairs pooled; neuron pairs (a) all off-diagonal pairs of identified neurons, zero where no synapse, (b) only pairs with at least one synapse; chemical counts (a) summed over both directions, (b) the larger direction. Gap junction counts are symmetric in the source. Expected head values: electrical 2.8% (2.52 to 3.08), chemical 0.5% (0.45 to 0.55).

In [ ]:
def animal_corr(d, which="all"):
    """Per-animal correlation matrices on a common name index; which = all, stim or nonstim frames."""
    names = sorted({n for a in d["animals"] for n in a["traces"]} & set(chem.index) & set(gap.index))
    ix = {n: i for i, n in enumerate(names)}
    mats = []
    for a in d["animals"]:
        nm = sorted(set(a["traces"]) & set(ix))
        X = np.array([a["traces"][n] for n in nm]).T[4:]
        t = (np.arange(len(X)) + 4) / a["rate"]
        st = np.zeros(len(X), bool)
        for s0, s1 in a["stimuli"].values():
            st |= (t >= s0) & (t < s1)
        if which == "stim":
            X = X[st]
        elif which == "nonstim":
            X = X[~st]
        full = np.full((len(names), len(names)), np.nan)
        ii = [ix[n] for n in nm]
        full[np.ix_(ii, ii)] = np.corrcoef(X.T)
        mats.append(full)
    return names, np.array(mats)

def r2_table(names, mats, chem_m, gap_m, corr="pearson", logc=False, min_animals=0):
    iu = np.triu_indices(len(names), 1)
    G = gap_m.loc[names, names].values
    Cm = chem_m.loc[names, names].values
    syn = {"electrical": G[iu], "chemical (a) sum": (Cm + Cm.T)[iu], "chemical (b) max": np.maximum(Cm, Cm.T)[iu]}
    present = np.isfinite(mats[:, iu[0], iu[1]])
    navail = present.sum(0)
    with np.errstate(all="ignore"):
        fa = np.abs(np.nanmean(mats[:, iu[0], iu[1]], axis=0))
    out = {}
    for kind, cnt in syn.items():
        cnt = np.log1p(cnt) if logc else cnt
        for fread in ("a", "b"):
            for pread in ("a", "b"):
                keep = (navail > min_animals) & (navail > 0)
                if pread == "b":
                    keep &= cnt > 0
                if fread == "a":
                    f, c = fa[keep], cnt[keep]
                else:
                    sel = present[:, keep]
                    f = np.abs(mats[:, iu[0][keep], iu[1][keep]][sel])
                    c = np.broadcast_to(cnt[keep], sel.shape)[sel]
                r = stats.pearsonr(f, c)[0] if corr == "pearson" else stats.spearmanr(f, c)[0]
                out[(kind, f"functional ({fread})", f"pairs ({pread})")] = (100 * r ** 2, len(f))
    return out

cn_names, cn_mats = animal_corr(D, "all")
print("neurons in the comparison:", len(cn_names))
base = r2_table(cn_names, cn_mats, chem, gap)
tab = pd.DataFrame([{"synapses": k[0], "functional": k[1], "pairs": k[2], "R2_percent": v[0], "n": v[1]} for k, v in base.items()])
print(tab.round(3).to_string(index=False))
for mode in ("one", "none"):
    nm, mt = animal_corr(data[mode], "all")
    alt = r2_table(nm, mt, chem, gap)
    print(f"[{mode}]", {f"{k[0][:12]} {k[1][-3:]} {k[2][-3:]}": round(v[0], 3) for k, v in alt.items()})
chem5, gap5 = npl.load_cook(npl.COOK_ADJACENCY, "hermaphrodite chemical", "herm gap jn symmetric")
alt5 = r2_table(cn_names, cn_mats, chem5, gap5)
print("[SI 5 weighted adjacency, a variant]", {f"{k[0][:12]} {k[1][-3:]} {k[2][-3:]}": round(v[0], 3) for k, v in alt5.items()})

**Result: partially reproduced.** Every head R2 is below 5%, but none reaches the paper's values. Electrical synapses give 1.34% under readings (a, a) and 3.53% over pairs with a synapse (a, b), against 2.52 to 3.08%. Chemical synapses give 0.03 to 0.20%, against 0.45 to 0.55%. The other near-duplicate readings and the size-weighted SI 5 matrices give similar values (electrical 0.34 to 3.56%, chemical at most 0.29%). The tail values cannot be computed, as the dandiset holds no tails.

## fig6-connectome-variations

> Variations of the comparison (thresholding for low sampling, ranked correlation, log-scaled synapse counts, stimulus or non-stimulus periods only) do not noticeably improve the correlation between functional activity and the connectome.

Each variation on the readings of fig6jk-connectome-correlation (all readings are shown; the claims file asks for the reading that matched, or the first listed). Reproduced when no variation gives an R2 above 3.1%.

In [ ]:
var = {}
for th in (5, 10, 15):
    var[f"pairs in > {th} animals"] = r2_table(cn_names, cn_mats, chem, gap, min_animals=th)
var["Spearman"] = r2_table(cn_names, cn_mats, chem, gap, corr="spearman")
var["log(1 + count)"] = r2_table(cn_names, cn_mats, chem, gap, logc=True)
for which in ("stim", "nonstim"):
    nm, mt = animal_corr(D, which)
    var[f"{which} frames only"] = r2_table(nm, mt, chem, gap)
vt = pd.DataFrame([{"variation": v, "synapses": k[0], "functional": k[1], "pairs": k[2], "R2_percent": x[0], "n": x[1]}
                   for v, tbl in var.items() for k, x in tbl.items()])
print(vt.pivot_table(index=["synapses", "functional", "pairs"], columns="variation", values="R2_percent").round(3).to_string())
first = vt[(vt.functional == "functional (a)") & (vt.pairs == "pairs (a)") & (vt.synapses != "chemical (b) max")]
print("largest R2 under the first-listed readings: %.3f%% (%s)" % (first.R2_percent.max(),
      first.loc[first.R2_percent.idxmax(), ["variation", "synapses"]].tolist()))
print("largest R2 over all readings and variations: %.3f%%" % vt.R2_percent.max())

**Result: reproduced.** Under the first-listed readings, which the criterion uses because no reading matched in fig6jk-connectome-correlation, the largest R2 over all variations is 1.66% (electrical, pairs seen in more than 15 animals), below 3.1%. Over all readings, variations reach at most 3.64% (electrical, pairs with a synapse, log-scaled counts), against 3.53% for that reading without a variation.

## Claims not testable on this dandiset

The dandiset holds only the 21 OH16230 head recordings. The claims below rest on imaging the dandiset does not hold (the NeuroPAL strain characterisation, the positional atlas, receptor reporter strains, mutants, the 10 atlas worms of the identification software, and the optimal-coloring simulations). They are blocked as `data_missing`; the attempt record says, for each, what is missing.

In [ ]:
present = {"acquisition": sorted({k for r in recs[:1] for k in ("CalciumImageSeries", "NeuroPALImageRaw")}),
           "strains": sorted({r["strain"] for r in recs}), "descriptions": sorted({r["session_description"] for r in recs})}
print(present)
print("runtime so far: %.1f min" % ((time.time() - T_START) / 60))

## Summary

SUMMARY